清洗数据

In [1]:
# 全局设定

# after filter
# output_file = "experiment_result_cleanE0401.json"
output_file = "experiment_result_cleanE0502.json"
# output_file = "experiment_resultE0506-p.json"

# baseline
# output_file = "experiment_result_clean-blllmE0401.json"
# output_file = "experiment_result_clean-blllmE0502-p.json"
# output_file = "experiment_result_clean-blllmE0507-p.json"


有无catch必要的判断的计算

In [2]:
import json

experiment_data = []
with open(output_file, "r", encoding="utf-8") as f:
    # 读取 JSON 文件内容并解析为 Python 列表
    experiment_data = json.load(f)

def calculate_metrics(experiment_data):
    # 初始化变量
    true_positives = 0
    false_positives = 0
    false_negatives = 0
    true_negatives = 0
    
    for item in experiment_data:
        # 获取标签和预测值
        true_label = item.get("label", -1) 
        predicted_value = item.get("changed", 0)
        
        # # 跳过需要舍弃的数据
        # if true_label == -1:
        #     continue

        # if item.get("methodBefore").count('\n') < 6:
        #     continue

        # if item.get("methodBefore").startswith("@Test") or item.get("file_path").endswith("Test.java"):
        #     continue

        # if len(item.get("exceptionTypes")) == 1 and "RuntimeException" in item.get("exceptionTypes"):
        #     continue
        
        # 确定真实类别和预测类别
        true_class = 1 if true_label == 1 else 0
        predicted_class = 1 if predicted_value > 0 else 0
        
        # 更新统计量
        if true_class == 1 and predicted_class == 1:
            true_positives += 1
        elif true_class == 1 and predicted_class == 0:
            false_negatives += 1
        elif true_class == 0 and predicted_class == 1:
            false_positives += 1
        elif true_class == 0 and predicted_class == 0:
            true_negatives += 1
    
    # 计算指标
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        "total": true_positives + false_positives + false_negatives + true_negatives,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "true_negatives": true_negatives
    }


metrics = calculate_metrics(experiment_data)
print(metrics)
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1']:.4f}")

{'total': 1594, 'precision': 0.4649214659685864, 'recall': 0.8043478260869565, 'f1': 0.5892501658925017, 'true_positives': 444, 'false_positives': 511, 'false_negatives': 108, 'true_negatives': 531}
Precision: 0.4649
Recall: 0.8043
F1 Score: 0.5893


qwen

Precision: 0.4649
Recall: 0.8043
F1 Score: 0.5893
消融
Precision: 0.4106
Recall: 0.6991
F1 Score: 0.5173

deepseek

Precision: 0.5720
Recall: 0.5593
F1 Score: 0.5655
消融
Precision: 0.5623
Recall: 0.3926
F1 Score: 0.4624

EUNIE
Precision: 0.5587
Recall: 0.5435
F1 Score: 0.5510
消融
Precision: 0.5308
Recall: 0.5471
F1 Score: 0.5388

以行为单位计算嵌套行的计算

In [14]:
import json

experiment_data = []
with open(output_file, "r", encoding="utf-8") as f:
    # 读取 JSON 文件内容并解析为 Python 列表
    experiment_data = json.load(f)

def calculate_line_metrics(experiment_data):
    # 初始化变量
    true_positives_lines = 0
    false_positives_lines = 0
    false_negatives_lines = 0
    true_negatives_lines = 0
    
    for item in experiment_data:
        # # 跳过需要舍弃的数据
        # true_label = item.get("label", -1) 
        # if true_label == -1:
        #     continue
        # if item.get("methodBefore").count('\n') < 10:
        #     continue

        true_label = item.get("label", -1) 
        if true_label != 1:
            continue

        # 获取数据
        method_length = item.get("methodBefore").count('\n')
        true_lines = item.get("beforeTargetNoNestingLines")
        predict_lines = item.get("resultBeforeTargetNoNestingLines")

        true_positives = 0
        false_positives = 0
        false_negatives = 0
        true_negatives = 0

        for line_index in true_lines:
            if line_index in predict_lines:
                true_positives += 1
            else:
                false_negatives += 1
        false_positives += len(predict_lines) - true_positives
        true_negatives = method_length - true_positives - false_positives - false_negatives
        
        true_positives_lines += true_positives
        false_positives_lines += false_positives
        false_negatives_lines += false_negatives
        true_negatives_lines += true_negatives
    
    # 计算指标
    line_precision = true_positives_lines / (true_positives_lines + false_positives_lines) if (true_positives_lines + false_positives_lines) > 0 else 0
    line_recall = true_positives_lines / (true_positives_lines + false_negatives_lines) if (true_positives_lines + false_negatives_lines) > 0 else 0
    f1 = 2 * (line_precision * line_recall) / (line_precision + line_recall) if (line_precision + line_recall) > 0 else 0
    
    return {
        "line_precision": line_precision,
        "line_recall": line_recall,
        "line_f1": f1,
        "line_true_positives": true_positives_lines,
        "line_false_positives": false_positives_lines,
        "line_false_negatives": false_negatives_lines,
        "line_true_negatives": true_negatives_lines
    }

line_metrics = calculate_line_metrics(experiment_data)
print(line_metrics)
print(f"Line Precision: {line_metrics['line_precision']:.4f}")
print(f"Line Recall: {line_metrics['line_recall']:.4f}")
print(f"Line F1 Score: {line_metrics['line_f1']:.4f}")

    

{'line_precision': 0.3694459681843116, 'line_recall': 0.5513712648383136, 'line_f1': 0.44243718180325176, 'line_true_positives': 1347, 'line_false_positives': 2299, 'line_false_negatives': 1096, 'line_true_negatives': 10127}
Line Precision: 0.3694
Line Recall: 0.5514
Line F1 Score: 0.4424


In [ ]:
以行为单位不计算嵌套的计算

In [20]:
import json

experiment_data = []
with open(output_file, "r", encoding="utf-8") as f:
    # 读取 JSON 文件内容并解析为 Python 列表
    experiment_data = json.load(f)

def calculate_line_metrics(experiment_data):
    # 初始化变量
    true_positives_lines = 0
    false_positives_lines = 0
    false_negatives_lines = 0
    true_negatives_lines = 0
    
    for item in experiment_data:
        # # 跳过需要舍弃的数据
        # true_label = item.get("label", -1) 
        # if true_label == -1:
        #     continue
        # if item.get("methodBefore").count('\n') < 10:
        #     continue
        true_label = item.get("label", -1) 
        if true_label != 1:
            continue

        # 获取数据
        method_length = item.get("methodBefore").count('\n')
        # true_lines = item.get("beforeTargetNoNestingLines")
        true_lines = [i for i in range(item.get("beforeTargetStartLine"), item.get("beforeTargetEndLine"))]
        predict_lines = [i for i in range(item.get("resultBeforeTargetStartLine"), item.get("resultBeforeTargetEndLine"))]

        true_positives = 0
        false_positives = 0
        false_negatives = 0
        true_negatives = 0

        for line_index in true_lines:
            if line_index in predict_lines:
                true_positives += 1
            else:
                false_negatives += 1
        false_positives += len(predict_lines) - true_positives
        true_negatives = method_length - true_positives - false_positives - false_negatives
        
        true_positives_lines += true_positives
        false_positives_lines += false_positives
        false_negatives_lines += false_negatives
        true_negatives_lines += true_negatives
    
    # 计算指标
    line_precision = true_positives_lines / (true_positives_lines + false_positives_lines) if (true_positives_lines + false_positives_lines) > 0 else 0
    line_recall = true_positives_lines / (true_positives_lines + false_negatives_lines) if (true_positives_lines + false_negatives_lines) > 0 else 0
    f1 = 2 * (line_precision * line_recall) / (line_precision + line_recall) if (line_precision + line_recall) > 0 else 0
    
    return {
        "line_precision": line_precision,
        "line_recall": line_recall,
        "line_f1": f1,
        "line_true_positives": true_positives_lines,
        "line_false_positives": false_positives_lines,
        "line_false_negatives": false_negatives_lines,
        "line_true_negatives": true_negatives_lines
    }

line_metrics = calculate_line_metrics(experiment_data)
print(line_metrics)
print(f"Line Precision: {line_metrics['line_precision']:.4f}")
print(f"Line Recall: {line_metrics['line_recall']:.4f}")
print(f"Line F1 Score: {line_metrics['line_f1']:.4f}")

    

{'line_precision': 0.36820192557897474, 'line_recall': 0.5650958466453674, 'line_f1': 0.4458799432802899, 'line_true_positives': 1415, 'line_false_positives': 2428, 'line_false_negatives': 1089, 'line_true_negatives': 9937}
Line Precision: 0.3682
Line Recall: 0.5651
Line F1 Score: 0.4459


第一部预测正确的实例中，行和类型正确比例

In [4]:
import json

experiment_data = []
with open(output_file, "r", encoding="utf-8") as f:
    # 读取 JSON 文件内容并解析为 Python 列表
    experiment_data = json.load(f)

def calculate_metrics(experiment_data):
    # 初始化变量
    true_positives = 0
    line_match = 0
    type_match = 0
    false_negatives = 0
    only_runtime = 0
    num = 0
    
    for item in experiment_data:
        # 获取标签和预测值
        true_label = item.get("label", -1) 
        predicted_value = item.get("changed", 0)
        
        # # 跳过需要舍弃的数据
        # if true_label == -1:
        #     continue

        # if item.get("methodBefore").count('\n') < 10:
        #     continue
        
        # 确定真实类别和预测类别
        true_class = 1 if true_label == 1 else 0
        predicted_class = 1 if predicted_value > 0 else 0

        # if not (true_class == predicted_class == 1):
        #     continue

        true_lines = item.get("beforeTargetNoNestingLines")
        predict_lines = range(item.get("resultBeforeTargetStartLine"), item.get("resultBeforeTargetEndLine"))
        line_include = 1 if sum([(i not in predict_lines) for i in true_lines]) == 0 else 0

        true_exceptions = item.get("exceptionTypes")
        raw_predict_exceptions = item.get("resultExceptionTypes")
        predict_exceptions = []
        for e in raw_predict_exceptions:
            predict_exceptions += e.split('|')
        exception_include = 1 if sum([(i in predict_exceptions) for i in true_exceptions]) > 0 else 0

        if true_exceptions.__len__() == 1 and "RuntimeException" in true_exceptions:
            only_runtime += 1
        
        # 更新统计量
        if true_class == 1 and predicted_class == 1:
            true_positives += 1
            line_match += line_include
            type_match += exception_include
            if (line_include and exception_include):
                print(num)
                num += 1
                print("repoID, patch, methodName: ")
                print('"' + item.get("repo_id") + "\",\n\"" + item.get("patch") + "\",\n\"" + item.get("methodName") + '"')
        elif true_class == 1 and predicted_class == 0:
            false_negatives += 1
        
    print(only_runtime)

    # 计算指标
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    
    return {
        "recall": recall,
        "true_positives": true_positives,
        "false_negatives": false_negatives,
        "line_matches": line_match,
        "type_matches": type_match
    }


metrics = calculate_metrics(experiment_data)
print(metrics)
print(f"Recall: {metrics['recall']:.4f}")

0
repoID, patch, methodName: 
"FCL-Team/FoldCraftLauncher",
"<Patch - https://github.com/FCL-Team/FoldCraftLauncher/commit/40e0c330b29181c9727d757d70d80894bbce92ac>",
"getGuiScale(int,int,int)"
1
repoID, patch, methodName: 
"FCL-Team/FoldCraftLauncher",
"<Patch - https://github.com/FCL-Team/FoldCraftLauncher/commit/e5d4b03a76d51a0c87ac80b5e833e4e7920024ed>",
"applyBackground(Context,View,String,String)"
2
repoID, patch, methodName: 
"flyway/flyway",
"<Patch - https://github.com/flyway/flyway/commit/e436615927e3a7da7a0562c75aaecb728ef4880b>",
"toConfiguration(Map<String,Object>)"
3
repoID, patch, methodName: 
"flyway/flyway",
"<Patch - https://github.com/flyway/flyway/commit/50054634d4588ff71091c102ad369ab6e6ef5599>",
"toConfiguration(Map<String,Object>)"
4
repoID, patch, methodName: 
"flyway/flyway",
"<Patch - https://github.com/flyway/flyway/commit/567452972be2ed089086bc580da03216f60d0069>",
"getIntProperty(Properties,String,int)"
5
repoID, patch, methodName: 
"Anuken/Mindustry",
"<Pa

"Anuken/Mindustry",
"<Patch - https://github.com/Anuken/Mindustry/commit/d58aab499cf65901c015269a360062ad51a3a02f>",
"handleItem(Item,Tile,Tile)"
图漂亮

### 是它就是它
"SonarSource/sonarqube",
"<Patch - https://github.com/SonarSource/sonarqube/commit/f09de6aa56a805127e9f3c169b9fab7d50cd48fc>",
"getAuthenticationStatusPage(HttpServletRequest,HttpServletResponse)"
抓的对

"SonarSource/sonarqube",
"<Patch - https://github.com/SonarSource/sonarqube/commit/f4c27bdd408300ef9205b776beca442094cfb113>",
"submitTask(FileSystem,BlameResult,ExecutorService,InputFile)"
图复杂，抓对了，暂定它

38
repoID, patch, methodName: 
"sparrowwallet/sparrow",
"<Patch - https://github.com/sparrowwallet/sparrow/commit/9ec5b6ce266bfe49928c2802cd3d5f4e05551fac>",
"getSamouraiNetwork()"
没有树，但API有，fix error

47
repoID, patch, methodName: 
"M66B/FairEmail",
"<Patch - https://github.com/M66B/FairEmail/commit/b9d400e161cf695ad5218f177cfbbf493d82950b>",
"onActionTakePhoto()"
有树，匹配类型，Prevent crash，但异常来源为api

51
repoID, patch, methodName: 
"androidx/media",
"<Patch - https://github.com/androidx/media/commit/140e110e4410e6226a6411a5246e08a9c2d1fd91>",
"isInternetConnectivityValidated(ConnectivityManager)"
没树，其他没啥问题

61
repoID, patch, methodName: 
"iterate-ch/cyberduck",
"<Patch - https://github.com/iterate-ch/cyberduck/commit/6273b723b30ef5eef22f578bf49097b838ca3da6>",
"releaseShare(Path)"
光message对

71
repoID, patch, methodName: 
"iterate-ch/cyberduck",
"<Patch - https://github.com/iterate-ch/cyberduck/commit/72f5cc1073e4881433608ef3319eccce703abebb>",
"noop()"
图大，类型对但冗余，fix

81
repoID, patch, methodName: 
"jitsi/jitsi",
"<Patch - https://github.com/jitsi/jitsi/commit/dcb607dd8b10f297af6a590e29a7b9b6f6002aff>",
"createMediaDescriptionsForAnswer(SessionDescription)"
都挺好但捕获位置严重冗余

107
repoID, patch, methodName: 
"nextcloud/android",
"<Patch - https://github.com/nextcloud/android/commit/121a462bdf08707c2518a9c4a438d8d723c2fbf5>",
"uploadFile()"


第一步预测正确实例中，行覆盖率

In [2]:
import json

experiment_data = []
with open(output_file, "r", encoding="utf-8") as f:
    # 读取 JSON 文件内容并解析为 Python 列表
    experiment_data = json.load(f)

def calculate_line_metrics(experiment_data):
    # 初始化变量
    true_positives_lines = 0
    false_positives_lines = 0
    false_negatives_lines = 0
    true_negatives_lines = 0
    total_example = 0
    start_include = 0
    pnns, pnests, ns = 0, 0, 0
    pnn_ps, pnn_ns = 0, 0
    percentage = 0.5
    is_caught = 0
    
    for item in experiment_data:
        # 获取标签和预测值
        true_label = item.get("label", -1) 
        predicted_value = item.get("changed", 0)

        # 确定真实类别和预测类别
        true_class = 1 if true_label == 1 else 0
        predicted_class = 1 if predicted_value > 0 else 0

        if not (true_class == predicted_class == 1):
            continue
        total_example += 1

        # 获取数据
        method_length = item.get("methodBefore").count('\n')
        true_nn_lines = item.get("beforeTargetNoNestingLines")
        true_lines = [i for i in range(item.get("beforeTargetStartLine"), item.get("beforeTargetEndLine"))]
        predict_nn_lines = item.get("resultBeforeTargetNoNestingLines")
        predict_lines = [i for i in range(item.get("resultBeforeTargetStartLine"), item.get("resultBeforeTargetEndLine"))] + predict_nn_lines

        if true_nn_lines[0] in predict_lines:
            start_include += 1

        pnn_p, pnn_n, pnest_p, pnest_n, n_p, n_n = 0, 0, 0, 0, 0, 0

        for i in range(method_length):
            if i in true_nn_lines and i in predict_lines:
                pnn_p += 1
            elif i in true_nn_lines and i not in predict_lines:
                pnn_n += 1
            elif i in true_lines and i in predict_lines:
                pnest_p += 1
            elif i in true_lines and i not in predict_lines:
                pnest_n += 1
            elif i not in true_lines and i in predict_lines:
                n_p += 1
            elif i not in true_lines and i not in predict_lines:
                n_n += 1
        
        pnns += pnn_n + pnn_p
        pnests += pnest_n + pnest_p
        ns += n_n + n_p
        pnn_ps += pnn_p
        pnn_ns += pnn_n

        if pnn_p + pnn_n > 0 and pnn_p / (pnn_p + pnn_n) >= percentage and true_nn_lines[0] in predict_lines:
            is_caught += 1

        true_positives_lines += pnn_p + pnest_p
        false_positives_lines += n_p
        false_negatives_lines += pnn_n
        true_negatives_lines += pnest_n + n_n
    
    print(f"Pnn: {str(pnns)}, Pnest: {str(pnests)}, N: {str(ns)}")
    print(f"true positive: {str(pnn_ps)}, false negative: {str(pnn_ns)}")
    print(f"in total {str(total_example)} examples caught {str(is_caught)} .")
    
    # 计算指标
    line_precision = true_positives_lines / (true_positives_lines + false_positives_lines) if (true_positives_lines + false_positives_lines) > 0 else 0
    line_recall = true_positives_lines / (true_positives_lines + false_negatives_lines) if (true_positives_lines + false_negatives_lines) > 0 else 0
    f1 = 2 * (line_precision * line_recall) / (line_precision + line_recall) if (line_precision + line_recall) > 0 else 0
    
    return {
        "line_precision": line_precision,
        "line_recall": line_recall,
        "line_f1": f1,
        "line_true_positives": true_positives_lines,
        "line_false_positives": false_positives_lines,
        "line_false_negatives": false_negatives_lines,
        "line_true_negatives": true_negatives_lines,
        "total_example": total_example,
        "start_include": start_include
    }

line_metrics = calculate_line_metrics(experiment_data)
print(line_metrics)
print(f"Line Precision: {line_metrics['line_precision']:.4f}")
print(f"Line Recall: {line_metrics['line_recall']:.4f}")
print(f"Line F1 Score: {line_metrics['line_f1']:.4f}")

    

Pnn: 2073, Pnest: 178, N: 9053
true positive: 1367, false negative: 706
in total 444 examples caught 293 .
{'line_precision': 0.3726273726273726, 'line_recall': 0.6787989080982711, 'line_f1': 0.48113511770396644, 'line_true_positives': 1492, 'line_false_positives': 2512, 'line_false_negatives': 706, 'line_true_negatives': 6594, 'total_example': 444, 'start_include': 299}
Line Precision: 0.3726
Line Recall: 0.6788
Line F1 Score: 0.4811
